# TUDataset OHSU Graph Classification with GNNVisualizer

This notebook trains a graph-level GAT model on the PyTorch Geometric `TUDataset(name="OHSU")` benchmark, then renders the trained model with `GNNVisualizer`.

OHSU is a small bioinformatics/neuroscience graph classification dataset. The TU Dortmund dataset table reports an average of about 82 nodes and 200 edges per graph, so it is a good fit for testing mid-sized graph visualization around the 100-node scale.

Source docs: [PyG TUDataset](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.datasets.TUDataset.html) and [TU Dortmund graph datasets](https://chrsmrrs.github.io/datasets/docs/datasets/).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric
```

Optional environment variables: `OHSU_EPOCHS`, `OHSU_TARGET_NODES`, and `OHSU_HIDDEN_CHANNELS`.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

from gnn_exp import GNNVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("OHSU_EPOCHS", "80"))
TARGET_NODES = int(os.environ.get("OHSU_TARGET_NODES", "100"))
HIDDEN_CHANNELS = int(os.environ.get("OHSU_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = 16


def prepare_graph(data):
    data = data.clone()
    if getattr(data, "x", None) is None:
        degree = torch.bincount(data.edge_index[0], minlength=data.num_nodes).float().view(-1, 1)
        data.x = degree / degree.max().clamp_min(1.0)
    else:
        data.x = data.x.float()
    data.y = data.y.view(-1).long()
    return data


dataset = TUDataset(root=str(repo_root / "data" / "tudataset"), name="OHSU")
graphs = [prepare_graph(dataset[index]) for index in range(len(dataset))]
generator = torch.Generator().manual_seed(SEED)
order = torch.randperm(len(graphs), generator=generator).tolist()
train_size = max(1, int(0.8 * len(order)))
train_graphs = [graphs[index] for index in order[:train_size]]
valid_graphs = [graphs[index] for index in order[train_size:]] or train_graphs[:1]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)

visual_index = min(range(len(graphs)), key=lambda index: abs(graphs[index].num_nodes - TARGET_NODES))
visual_data = graphs[visual_index]
query_pair = [0, min(visual_data.num_nodes - 1, max(1, visual_data.num_nodes // 2))]
num_features = visual_data.num_features
num_classes = int(dataset.num_classes)

node_counts = torch.tensor([graph.num_nodes for graph in graphs], dtype=torch.float)
edge_counts = torch.tensor([graph.edge_index.size(1) for graph in graphs], dtype=torch.float)
display(Markdown(
    f"OHSU loaded with **{len(dataset)} graphs**. "
    f"This notebook trains on **{len(train_graphs)} graphs** and validates on **{len(valid_graphs)} graphs**. "
    f"Mean graph size in the local copy is **{node_counts.mean():.1f} nodes** and **{edge_counts.mean():.1f} directed edges**. "
    f"The visualized graph is index `{visual_index}` with **{visual_data.num_nodes} nodes**, "
    f"**{visual_data.edge_index.size(1)} directed edges**, and **{num_features} node features**."
))

In [ ]:
class OHSUGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("hidden_channels must be divisible by 2")
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch)
            pred = logits.argmax(dim=1)
            target = batch.y.view(-1).long()
            correct += int((pred == target).sum())
            total += int(target.numel())
    return correct / max(total, 1)


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    losses = []
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(logits, batch.y.view(-1).long())
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
    return {
        "final_loss": losses[-1],
        "train_accuracy": accuracy(model, train_loader),
        "valid_accuracy": accuracy(model, valid_loader),
    }


torch.manual_seed(SEED)
model = OHSUGAT(num_features, HIDDEN_CHANNELS, num_classes)
metrics = train_model(model, train_loader)
model.eval()

display(Markdown(
    "| Metric | Value |\n"
    "|---|---:|\n"
    f"| Final training loss | {metrics['final_loss']:.4f} |\n"
    f"| Train accuracy | {metrics['train_accuracy']:.3f} |\n"
    f"| Validation accuracy | {metrics['valid_accuracy']:.3f} |"
))

The next cell builds the widget. `mode="graph"` lets `GNNVisualizer` capture the message-passing layers, classifier output, and the graph-level `global_mean_pool` readout.

In [ ]:
visualizer = GNNVisualizer(renderer="svg")
visualizer.add_model(
    data=visual_data,
    model=model,
    subgraphSample=False,
    queries=[query_pair],
    mode="graph",
)

assert visualizer.modelInfo["conv1"]["type"] == "GATConv"
assert visualizer.modelInfo["conv1"].get("aggregation") == "attention"
assert len(visualizer.graphData["x"]) == visual_data.num_nodes
assert "graphAggregation" in visualizer.intmData
assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS

display(Markdown(
    "| Captured item | Value |\n"
    "|---|---:|\n"
    f"| First layer | `{visualizer.modelInfo['conv1']['type']}` |\n"
    f"| Aggregation | `{visualizer.modelInfo['conv1'].get('aggregation')}` |\n"
    f"| Graph pooling | `{visualizer.intmData['graphAggregation']['type']}` |\n"
    f"| Hidden width | {len(visualizer.intmData['act1'][0])} |\n"
    f"| Visualized nodes | {len(visualizer.graphData['x'])} |\n"
    f"| Query | `{visualizer.queries}` |"
))

In [ ]:
display(visualizer)